###Intro to neural network classigication using tensorflow


In [ ]:
from sklearn.datasets import make_circles

# make 1000 examples
n_samples = 1000

# create circles
x, y = make_circles(n_samples,
                    noise = 0.03)

In [ ]:
# check out the features
x # x is a 2D array that contains 2 coordinates

In [ ]:
# check the labesl
y

Visualizing our data

In [ ]:
import pandas as pd
circles = pd.DataFrame({"X0":x[:,0], "x1":x[:,1], "label":y}) # x0 is the 1st element and x1 is the 2nd element
circles


In [ ]:
import matplotlib.pyplot as plt
plt.scatter(x[:,0], x[:,1], c = y, cmap = plt.cm.RdYlBu)

# c = y is to color each dot based on its label in the y array
# RdYlBu: Stands for Red-Yellow-Blue.


In [ ]:
x.shape, y.shape

In [ ]:
len(x), len(y)

In [ ]:
x[0], y[0]

Build our Model

In [ ]:
import tensorflow as tf

In [ ]:
model_1 = tf.keras.Sequential([
    tf.keras.layers.Dense(1)
])
model_1.compile(loss = ["BinaryCrossentropy"],
                optimizer = "SGD", # no square brackets since optimizer expects a single value not a list
                metrics = ["accuracy"]) # accuracy is: how many did we get right in a certain batch size
model_1.fit(x,y,epochs = 5, verbose = 0)

model_1.evaluate(x,y)


In [ ]:
# try improving by training for longer

model_1.fit(x,y, epochs=200,verbose =0)
model_1.evaluate(x,y) # ideally we evaluate using testing data not training data

In [ ]:
# add an extra layer

model_2 = tf.keras.Sequential([
    tf.keras.layers.Dense(1),
    tf.keras.layers.Dense(1)
])
model_2.compile(loss = ["BinaryCrossentropy"],
                optimizer = "SGD", # no square brackets since optimizer expects a single value not a list
                metrics = ["accuracy"]) # accuracy is: how many did we get right in a certain batch size
model_2.fit(x,y,epochs = 200, verbose=0)

In [ ]:
model_2.evaluate(x,y)

# despite adding a layer, accuracy has not improved

Improving our Model

In [ ]:
'''
1. create a model - add, increase layers
2. compile - change optimizer (use adam?), change activisation funct, change learning_rate
3. fit - fit for longer
'''

In [ ]:
model_3 = tf.keras.Sequential([
    tf.keras.layers.Dense(100),
    tf.keras.layers.Dense(10),
    tf.keras.layers.Dense(1)
])

# Create an Adam optimizer with a custom learning rate
adam_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model_3.compile(loss = "BinaryCrossentropy",
                optimizer = tf.keras.optimizers.Adam(), # Pass the optimizer object here
                metrics = ["accuracy"])

model_3.fit(x,y,epochs = 100, verbose=0)

In [ ]:
model_3.evaluate(x,y)
# even with all these changes the model accuracy has not increased

In [ ]:
'''
create a meshgrid of the different X values
make predictions across the meshgrid
plot the model's predictions as well as the line between zones
'''

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# create our function
def plot_decision_boundary(model,x,y):
  # plots the decision boundary created by a model predicting on x.

  # define axis boundaries of plot and create meshgrid
  x_min, x_max = x[:, 0].min()-0.1, x[:,0].max()+0.1 # 0.1 is to have breathing room for the min and max points
  y_min, y_max = x[:, 1].min()-0.1, x[:,1].max()+0.1 # Corrected y_min, y_max to use x[:,1]

  # Create a meshgrid from both linspace arrays
  xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

  # create x values to make predictions on
  x_in = np.c_[xx.ravel(), yy.ravel()] # np.c_ is a shorthand for the actual funct

  # Make predictions (these will be logits since the last layer is Dense(1) without activation)
  y_pred = model.predict(x_in)

  # check for multi-class (multiple labes for a single input)
  if len(y_pred[0]) > 1:
    print("doing multiclass classification")
    # we have to reshape our predictions to get it ready for plotting
    y_pred = np.argmax(y_pred,axis=1).reshape(xx.shape)
  else:
    print("doing binary classification")
    y_pred = np.round(y_pred).reshape(xx.shape)

  plot_data = y_pred

  # plot the decision boundary
  plt.contourf(xx,yy, plot_data.reshape(xx.shape), cmap = plt.cm.RdYlBu, alpha = 0.7)
  plt.scatter(x[:,0], x[:,1], c = y, s = 40, cmap = plt.cm.RdYlBu)
  plt.xlim(xx.min(), xx.max())
  plt.ylim(yy.min(), yy.max())
  plt.show() # Ensure plot is displayed

# Because your model only has linear layers, it can only draw a straight line through the data


In [ ]:
plot_decision_boundary(model_3, x, y=y)

In [ ]:
# Let's see if our model can be used for a regression problem

# create data
x_regression = tf.range(0,1000,5)
y_regression = x_regression + 100

x_regression, y_regression

In [ ]:
# split into trining and test set
x_reg_train = x_regression[:150]
x_reg_test = x_regression[150:] # last 150 values

y_reg_train = y_regression[:150]
y_reg_test = y_regression[150:]

In [ ]:
# Fit our model to the regression data

# model_3.fit(tf.expand_dims(x_reg_train), y_reg_train, epochs = 100)

### Model_3 was made with binary data in mind. It will not work when loss is set to BinaryCrossEntropy
### Make a new model with settigns loss to MAE

In [ ]:
# similar to model_3 but with MAE as loss instead of BinaryCrossEntropy (BCE is for binary data)

model_4 = tf.keras.Sequential([
    tf.keras.layers.Dense(100, input_shape=[1]), # Explicitly set input_shape to (1,)
    tf.keras.layers.Dense(10),
    tf.keras.layers.Dense(1)
])

# Create an Adam optimizer with a custom learning rate
adam_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model_4.compile(loss = "MAE",
                optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01), # Pass the optimizer object here
                metrics = ["MAE"])

In [ ]:
# fit model_4
model_4.fit(tf.expand_dims(x_reg_train, axis = -1),y_reg_train,epochs = 100, verbose=0)

In [ ]:
# Make predictions
y_reg_preds = model_4.predict(x_reg_test)

# pot the model's predictions
plt.figure(figsize = (10,7))
plt.scatter(x_reg_train, y_reg_train, c = "b", label = "Training Data")
plt.scatter(x_reg_test, y_reg_test, c = "g", label = "Testing Data")
plt.scatter(x_reg_test, y_reg_preds, c = "r", label = "Predictions")
plt.legend();


# Our model works for regression problem!
# => we need to introduce non-linearity to our model